# 03. Statistical Analysis & Hypothesis Testing
**UPI Fraud Detection and Risk Analytics**

### Objectives:
1. Conduct distribution normality testing (Shapiro-Wilk, Kolmogorov-Smirnov, Skewness, Kurtosis).
2. Perform two-sample hypothesis testing (Mann-Whitney U Test, Two-Sample t-test) to compare financial and latency metrics across Fraud vs Non-Fraud transactions.
3. Conduct Chi-Square (\(\chi^2\)) Tests of Independence for categorical features (`UPI_App`, `Payment_Method`, `Transaction_Type`, `Transaction_Status`).
4. Calculate Cramér's V effect sizes for categorical association.
5. Compute Odds Ratios and Relative Risk with 95% Confidence Intervals for key risk indicators.


In [1]:
import os
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
%matplotlib inline


## 1. Load Processed Dataset

In [2]:
data_path =  r"C:\UPI Data\UPI_Transactions_Cleaned.csv"
if not os.path.exists(data_path):
    data_path =  r"C:\UPI Data\UPI_Transactions_Cleaned.csv"

df = pd.read_csv(data_path)
print(f'Dataset Loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()


Dataset Loaded: 10000 rows, 24 columns


,Transaction_ID,Transaction_Date,Customer_ID,City,State,Sender_Bank,Receiver_Bank,UPI_App,Transaction_Type,Payment_Method,Amount,Transaction_Status,Failure_Reason,Processing_Time_sec,Bank_Response_Time_ms,Fraud_Label,Year,Month,Month_Name,Day,Day_of_Week,Day_of_Week_Num,Is_Weekend,Quarter
0,TXN100000,2025-11-24,CUST09196,Bengaluru,Karnataka,SBI,Kotak Mahindra Bank,PhonePe,Online Purchase,Mobile Number,2014.53,Success,Not Applicable,4.83,801,0,2025,11,Nov,24,Monday,0,0,4
1,TXN100001,2025-04-23,CUST01520,Ahmedabad,Gujarat,Yes Bank,Kotak Mahindra Bank,BHIM,P2P,UPI ID,1359.97,Success,Not Applicable,1.58,751,0,2025,4,Apr,23,Wednesday,2,0,2
2,TXN100002,2025-06-26,CUST09460,Lucknow,Uttar Pradesh,Kotak Mahindra Bank,SBI,PhonePe,P2P,UPI,8199.06,Success,Not Applicable,3.03,816,0,2025,6,Jun,26,Thursday,3,0,2
3,TXN100003,2025-12-05,CUST05821,Mumbai,Maharashtra,Kotak Mahindra Bank,HDFC Bank,PhonePe,Bill Payment,UPI,2240.06,Success,Not Applicable,3.58,345,0,2025,12,Dec,5,Friday,4,0,4
4,TXN100004,2025-11-08,CUST00526,Kochi,Kerala,ICICI Bank,Union Bank,Paytm,Recharge,UPI ID,320.36,Success,Not Applicable,4.48,888,0,2025,11,Nov,8,Saturday,5,1,4


## 2. Distribution Normality & Summary Metrics

In [3]:
num_cols = ['Amount', 'Processing_Time_sec', 'Bank_Response_Time_ms']
normality_results = []

for col in num_cols:
    skew = df[col].skew()
    kurt = df[col].kurtosis()
    # Sample 1000 points for Shapiro-Wilk test
    sample_data = df[col].sample(n=1000, random_state=42)
    shapiro_stat, shapiro_p = stats.shapiro(sample_data)
    ks_stat, ks_p = stats.ks_2samp(df[col], stats.norm.rvs(loc=df[col].mean(), scale=df[col].std(), size=len(df), random_state=42))
    
    normality_results.append({
        'Feature': col,
        'Mean': round(df[col].mean(), 2),
        'Std_Dev': round(df[col].std(), 2),
        'Skewness': round(skew, 3),
        'Kurtosis': round(kurt, 3),
        'Shapiro_Stat': round(shapiro_stat, 4),
        'Shapiro_p_value': round(shapiro_p, 4),
        'Is_Normal (p>0.05)': 'Yes' if shapiro_p > 0.05 else 'No (Skewed)'
    })

pd.DataFrame(normality_results)


,Feature,Mean,Std_Dev,Skewness,Kurtosis,Shapiro_Stat,Shapiro_p_value,Is_Normal (p>0.05)
0,Amount,1339.11,1836.86,8.112,170.265,0.5562,0.0000,No (Skewed)
1,Processing_Time_sec,3.99,2.73,1.555,4.448,0.9006,0.0000,No (Skewed)
2,Bank_Response_Time_ms,652.55,246.36,0.079,-0.153,0.9973,0.1011,Yes


## 3. Two-Sample Hypothesis Testing (Fraud vs Non-Fraud)

In [4]:
mwu_results = []

legit = df[df['Fraud_Label'] == 0]
fraud = df[df['Fraud_Label'] == 1]

for col in num_cols:
    legit_vals = legit[col]
    fraud_vals = fraud[col]
    
    # Mann-Whitney U test (Non-parametric)
    u_stat, p_val_mwu = stats.mannwhitneyu(legit_vals, fraud_vals, alternative='two-sided')
    
    # Welch's t-test (Parametric)
    t_stat, p_val_ttest = stats.ttest_ind(legit_vals, fraud_vals, equal_var=False)
    
    # Cohen's d effect size
    n1, n2 = len(legit_vals), len(fraud_vals)
    s1, s2 = legit_vals.std(), fraud_vals.std()
    s_pooled = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
    cohen_d = (fraud_vals.mean() - legit_vals.mean()) / s_pooled
    
    mwu_results.append({
        'Feature': col,
        'Legit_Mean': round(legit_vals.mean(), 2),
        'Fraud_Mean': round(fraud_vals.mean(), 2),
        'Mann_Whitney_U': u_stat,
        'MWU_p_value': p_val_mwu,
        'Welch_t_stat': round(t_stat, 3),
        'Welch_p_value': p_val_ttest,
        'Statistically_Significant': 'Yes' if p_val_mwu < 0.05 else 'No',
        'Cohens_d': round(cohen_d, 3)
    })

pd.DataFrame(mwu_results)


,Feature,Legit_Mean,Fraud_Mean,Mann_Whitney_U,MWU_p_value,Welch_t_stat,Welch_p_value,Statistically_Significant,Cohens_d
0,Amount,1340.73,1177.88,504831.0,0.606274,1.169,0.244952,No,-0.089
1,Processing_Time_sec,3.98,4.42,478108.5,0.674838,-1.177,0.241975,No,0.161
2,Bank_Response_Time_ms,652.16,692.11,442041.0,0.092680,-1.622,0.108039,No,0.162


## 4. Chi-Square Test of Independence (\(\chi^2\)) & Cramér's V

In [5]:
cat_cols = ['UPI_App', 'Transaction_Type', 'Payment_Method', 'Transaction_Status', 'Sender_Bank']
chi2_results = []

def cramers_v(contingency_table):
    chi2 = stats.chi2_contingency(contingency_table)[0]
    n = contingency_table.sum().sum()
    r, k = contingency_table.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1)))

for col in cat_cols:
    contingency = pd.crosstab(df[col], df['Fraud_Label'])
    chi2, p_val, dof, _ = stats.chi2_contingency(contingency)
    c_v = cramers_v(contingency)
    
    chi2_results.append({
        'Categorical_Feature': col,
        'Chi2_Statistic': round(chi2, 3),
        'Degrees_of_Freedom': dof,
        'p_value': p_val,
        'Cramers_V': round(c_v, 4),
        'Independent (p>0.05)': 'Yes' if p_val > 0.05 else 'No (Dependent)'
    })

pd.DataFrame(chi2_results)


,Categorical_Feature,Chi2_Statistic,Degrees_of_Freedom,p_value,Cramers_V,Independent (p>0.05)
0,UPI_App,2.767,5,7.357938e-01,0.0166,Yes
1,Transaction_Type,4.058,4,3.982677e-01,0.0201,Yes
2,Payment_Method,8.801,4,6.626829e-02,0.0297,Yes
3,Transaction_Status,35.384,2,2.072121e-08,0.0595,No (Dependent)
4,Sender_Bank,5.178,9,8.185643e-01,0.0228,Yes


## 5. Kruskal-Wallis Test Across Banks

In [6]:
bank_groups = [group['Bank_Response_Time_ms'].values for name, group in df.groupby('Sender_Bank')]
h_stat, kw_p_val = stats.kruskal(*bank_groups)

print(f'Kruskal-Wallis H-statistic for Bank Response Time across Sender Banks: {h_stat:.3f}')
print(f'p-value: {kw_p_val:.5e}')
print('Significant difference in response time across banks:', 'Yes' if kw_p_val < 0.05 else 'No')


Kruskal-Wallis H-statistic for Bank Response Time across Sender Banks: 7.494
p-value: 5.85854e-01
Significant difference in response time across banks: No


## 6. Odds Ratio & Relative Risk Analysis

In [7]:
df['High_Amount_Flag'] = (df['Amount'] > df['Amount'].quantile(0.75)).astype(int)
df['High_Latency_Flag'] = (df['Bank_Response_Time_ms'] > 800).astype(int)

risk_flags = ['High_Amount_Flag', 'High_Latency_Flag', 'Is_Weekend']
odds_summary = []

for flag in risk_flags:
    table = pd.crosstab(df[flag], df['Fraud_Label'])
    # Table layout: [[a, b], [c, d]] where row=flag (0, 1), col=fraud (0, 1)
    a = table.loc[1, 1]  # Flag=1, Fraud=1
    b = table.loc[1, 0]  # Flag=1, Fraud=0
    c = table.loc[0, 1]  # Flag=0, Fraud=1
    d = table.loc[0, 0]  # Flag=0, Fraud=0
    
    odds_ratio = (a * d) / (b * c) if (b * c) > 0 else np.nan
    se_log_or = np.sqrt(1/a + 1/b + 1/c + 1/d)
    ci_lower = np.exp(np.log(odds_ratio) - 1.96 * se_log_or)
    ci_upper = np.exp(np.log(odds_ratio) + 1.96 * se_log_or)
    
    relative_risk = (a / (a + b)) / (c / (c + d))
    
    odds_summary.append({
        'Risk_Indicator': flag,
        'Odds_Ratio': round(odds_ratio, 3),
        '95% CI Lower': round(ci_lower, 3),
        '95% CI Upper': round(ci_upper, 3),
        'Relative_Risk': round(relative_risk, 3)
    })

pd.DataFrame(odds_summary)


,Risk_Indicator,Odds_Ratio,95% CI Lower,95% CI Upper,Relative_Risk
0,High_Amount_Flag,0.960,0.605,1.524,0.961
1,High_Latency_Flag,1.280,0.838,1.954,1.276
2,Is_Weekend,1.648,1.098,2.472,1.639
